In [3]:
# ============================================================
# INGEST RESULTS — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType, DateType
)

In [4]:
# -----------------------------------------
# 1. IMPORT ENVIRONMENT CONFIG
# -----------------------------------------
try:
    BRONZE_LANDING
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    config_path = os.path.join(project_root, "/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 23:08:00 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/08 23:08:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 23:08:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 23:08:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/08 23:08:01 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/08 23:08:01 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/08 23:08:01 

✔ Spark Session Initialized
✔ Environment Config Loaded
PROJECT_ROOT: /Users/manoharazalki/F1-ANALYTICS


In [ ]:
# -----------------------------------------
# 2. IMPORT HELPER FUNCTIONS
# -----------------------------------------
try:
    add_ingestion_metadata
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    helper_path = "/Users/manoharazalki/F1-Analytics/notebooks/00-common/02_bronze_helpers.ipynb"
    with open(helper_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

✔ Bronze helper functions loaded


In [6]:
# -----------------------------------------
# 3. Define source + target paths
# -----------------------------------------
source_folder = f"{BRONZE_LANDING}/results"
target_folder = f"{BRONZE_ROOT}/results"
target_file = f"{target_folder}/results.parquet"

os.makedirs(target_folder, exist_ok=True)

print("Source:", source_folder)
print("Target:", target_file)

Source: /Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/results
Target: /Users/manoharazalki/F1-ANALYTICS/data/bronze/results/results.parquet


In [7]:
# -----------------------------------------
# 4. Define schema (same as Databricks)
# -----------------------------------------
result_schema = StructType([
    StructField("date", DateType(), True),
    StructField("raceName", StringType(), True),
    StructField("round", IntegerType(), True),
    StructField("season", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("constructorId", StringType(), True),
    StructField("driverId", StringType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("points", FloatType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("status", StringType(), True)
])

In [8]:
# -----------------------------------------
# 5. Read JSON (multiple files)
# -----------------------------------------
results_df = (
    spark.read
        .format("json")
        .schema(result_schema)
        .option("mode", "FAILFAST")
        .load(source_folder)
)

print("✔ Results files read successfully")
results_df.show(5, truncate=False)

✔ Results files read successfully
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+
|date      |raceName             |round|season|url                                                     |constructorId|driverId      |grid|laps|number|points|position|positionText|status  |
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+
|2025-03-16|australian grand prix|1    |2025  |https://en.wikipedia.org/wiki/2025_Australian_Grand_Prix|mclaren      |norris        |1   |57  |4     |25.0  |1       |1           |Finished|
|2025-03-16|australian grand prix|1    |2025  |https://en.wikipedia.org/wiki/2025_Australian_Grand_Prix|red_bull     |max_verstappen|3   |57  |1     |18.0  |2       |2           |Finished|
|2025-03-16|australia

In [9]:
# -----------------------------------------
# 6. Add metadata
# -----------------------------------------
results_final_df = add_ingestion_metadata(results_df, source_folder)

print("✔ Metadata added")
results_final_df.show(5, truncate=False)

✔ Metadata added
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+--------------------------+-------------------------------------------------------------+
|date      |raceName             |round|season|url                                                     |constructorId|driverId      |grid|laps|number|points|position|positionText|status  |ingest_timestamp          |source_file                                                  |
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+--------------------------+-------------------------------------------------------------+
|2025-03-16|australian grand prix|1    |2025  |https://en.wikipedia.org/wiki/2025_Australian_Grand_Prix|mclaren      |norris        |1   |57  |4     

In [10]:
# -----------------------------------------
# 7. Write to Bronze (Option C)
# -----------------------------------------
(
    results_final_df
        .write
        .mode("overwrite")
        .parquet(target_file)
)

print(f"✔ Results Bronze written to: {target_file}")

26/06/08 23:08:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/08 23:08:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/08 23:08:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers


✔ Results Bronze written to: /Users/manoharazalki/F1-ANALYTICS/data/bronze/results/results.parquet


26/06/08 23:08:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/08 23:08:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [11]:
# -----------------------------------------
# 8. Read back for validation
# -----------------------------------------
spark.read.parquet(target_file).show(5, truncate=False)

+----------+--------------------+-----+------+-------------------------------------------------------+-------------+------------------+----+----+------+------+--------+------------+--------+--------------------------+-------------------------------------------------------------+
|date      |raceName            |round|season|url                                                    |constructorId|driverId          |grid|laps|number|points|position|positionText|status  |ingest_timestamp          |source_file                                                  |
+----------+--------------------+-----+------+-------------------------------------------------------+-------------+------------------+----+----+------+------+--------+------------+--------+--------------------------+-------------------------------------------------------------+
|1995-03-26|brazilian grand prix|1    |1995  |https://en.wikipedia.org/wiki/1995_Brazilian_Grand_Prix|benetton     |michael_schumacher|2   |71  |1     |10.0  |1